# Steel Industry Energy Consumption - Modeling & Evaluation


This notebook implements the full modeling workflow:
1. **Load Splits** - Load train/validation/calibration/test CSV files
2. **Train Baselines** - Persistence and Historical Mean baselines
3. **Train Ridge Regression** - Linear reference model
4. **Train LightGBM** - Primary model with Huber loss
5. **Calibrate Alert Policy** - 90th percentile residual threshold
6. **Test & Audit** - Final evaluation metrics

In [1]:
#!/usr/bin/env python3
# Steel Industry Energy Consumption - Modeling & Evaluation
# Steps: 1. Load CSV splits, 2. Train baselines, 3. Ridge, 4. LightGBM, 5. Calibrate alert, 6. Evaluate

import json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
import lightgbm as lgb

warnings.filterwarnings('ignore')

# 1. Distinct paths for input datasets and output artifacts
DATA_DIR = Path("../data/steel/processed")
ARTIFACTS_DIR = Path("../notebooks/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Step 1: Load Splits & Metadata from data folder
print("Step 1: Loading CSV Splits and Metadata")
train_df = pd.read_csv(DATA_DIR / "steel_next_60m_train.csv")
val_df = pd.read_csv(DATA_DIR / "steel_next_60m_validation.csv")
cal_df = pd.read_csv(DATA_DIR / "steel_next_60m_calibration.csv")
test_df = pd.read_csv(DATA_DIR / "steel_next_60m_test.csv")

with open(DATA_DIR / "steel_quality_summary.json", "r") as f:
    quality_info = json.load(f)

ALERT_THRESHOLD = quality_info.get("train_target_p95_kWh", 379.771)
BASE_FEATURES = quality_info.get("features_base_list", [
    "usage_lag_0", "usage_lag_1", "usage_lag_4", "usage_lag_96", "usage_lag_672",
    "usage_sum_last_1h", "usage_mean_last_4h", "usage_std_last_4h",
    "usage_mean_last_24h", "usage_max_last_24h",
    "forecast_hour_sin", "forecast_hour_cos",
    "forecast_weekday_sin", "forecast_weekday_cos",
    "forecast_is_weekend"
])
ALL_FEATURES = BASE_FEATURES
TARGET = "target_next_60m_kWh"

print(f"Train: {len(train_df):,}, Val: {len(val_df):,}, Cal: {len(cal_df):,}, Test: {len(test_df):,}")

# Step 2: Persistence Baseline (Validation)
print("Step 2: Establishing Baselines")
val_persist_mae = mean_absolute_error(val_df[TARGET], val_df['usage_sum_last_1h'])
print(f"Persistence MAE (Val): {val_persist_mae:.2f} kWh")

# Step 3: Train Ridge Regression
print("Step 3: Training Ridge Regression")
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[ALL_FEATURES].values)
X_val = scaler.transform(val_df[ALL_FEATURES].values)

ridge = Ridge(alpha=1.0, random_state=42).fit(X_train, train_df[TARGET])
ridge_mae = mean_absolute_error(val_df[TARGET], ridge.predict(X_val))
print(f"Ridge MAE (Val): {ridge_mae:.2f} kWh")

# Step 4: Train LightGBM Regressor (L1 / MAE objective)
print("Step 4: Training LightGBM")
train_data = lgb.Dataset(train_df[ALL_FEATURES].values, label=train_df[TARGET])
val_data = lgb.Dataset(val_df[ALL_FEATURES].values, label=val_df[TARGET], reference=train_data)

params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'learning_rate': 0.03,
    'num_leaves': 31,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'verbose': -1,
    'seed': 42
}

start = time.time()
model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)
training_time = time.time() - start

# Save model file into artifacts directory
model_path = ARTIFACTS_DIR / "lightgbm_model.txt"
model.save_model(str(model_path))

lgb_pred_val = model.predict(val_df[ALL_FEATURES].values)
lgb_mae = mean_absolute_error(val_df[TARGET], lgb_pred_val)
print(f"LightGBM MAE (Val): {lgb_mae:.2f} kWh, Time: {training_time:.2f}s")
print(f"Model artifact saved to: {model_path}")

# Step 5: Calibrate Alert Policy on October Set
print("Step 5: Calibrating Alert Policy")
cal_pred = model.predict(cal_df[ALL_FEATURES].values)
e_residuals = np.maximum(cal_df[TARGET].values - cal_pred, 0)
q_90 = float(np.percentile(e_residuals, 90))

print(f"90th percentile under-forecast residual (q): {q_90:.2f} kWh")
print(f"Alert triggers if prediction + {q_90:.2f} > {ALERT_THRESHOLD:.3f} kWh")

# Step 6: Final Evaluation on Test Set (Row-level)
print("Step 6: Final Evaluation on Test Set")
test_pred = model.predict(test_df[ALL_FEATURES].values)
test_mae = mean_absolute_error(test_df[TARGET], test_pred)
test_rmse = mean_squared_error(test_df[TARGET], test_pred, squared=False)
print(f"Test MAE: {test_mae:.2f} kWh, RMSE: {test_rmse:.2f} kWh")

alerts = (test_pred + q_90) > ALERT_THRESHOLD
actual = test_df[TARGET].values > ALERT_THRESHOLD

precision = np.sum(alerts & actual) / max(np.sum(alerts), 1)
recall = np.sum(alerts & actual) / max(np.sum(actual), 1)
f1 = 2 * precision * recall / max(precision + recall, 1e-10)
print(f"Row-Level Alert: Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

results = {
    'test_mae_kWh': test_mae,
    'test_rmse_kWh': test_rmse,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'alerts_triggered': int(alerts.sum()),
    'actual_exceedances': int(actual.sum()),
    'training_time_s': training_time,
    'q_buffer_kWh': q_90
}

Step 1: Loading CSV Splits and Metadata
Train: 22,652, Val: 2,876, Cal: 2,972, Test: 5,852
Step 2: Establishing Baselines
Persistence MAE (Val): 32.32 kWh
Step 3: Training Ridge Regression
Ridge MAE (Val): 34.68 kWh
Step 4: Training LightGBM
LightGBM MAE (Val): 14.96 kWh, Time: 2.06s
Model artifact saved to: ..\notebooks\artifacts\lightgbm_model.txt
Step 5: Calibrating Alert Policy
90th percentile under-forecast residual (q): 42.35 kWh
Alert triggers if prediction + 42.35 > 379.771 kWh
Step 6: Final Evaluation on Test Set
Test MAE: 15.92 kWh, RMSE: 32.58 kWh
Row-Level Alert: Precision=0.5792, Recall=0.7151, F1=0.6400


In [2]:
# Step 7: Diurnal Mean Baseline (Historical by Hour & Day of Week)
print("Step 7: Computing Diurnal Mean Baseline")

def extract_hour_and_dow(df):
    """Reconstruct exact hour (0-23) and day-of-week (0-6) from cyclical sine/cos."""
    h_angle = np.arctan2(df["forecast_hour_sin"], df["forecast_hour_cos"])
    hour = np.mod(np.round(h_angle * 12.0 / np.pi), 24).astype(int)
    
    w_angle = np.arctan2(df["forecast_weekday_sin"], df["forecast_weekday_cos"])
    dow = np.mod(np.round(w_angle * 7.0 / (2.0 * np.pi)), 7).astype(int)
    return hour, dow

train_df["hour"], train_df["dow"] = extract_hour_and_dow(train_df)
val_df["hour"], val_df["dow"] = extract_hour_and_dow(val_df)
test_df["hour"], test_df["dow"] = extract_hour_and_dow(test_df)

# Group strictly from training data
diurnal_lookup = train_df.groupby(["dow", "hour"])[TARGET].mean().to_dict()
global_train_mean = train_df[TARGET].mean()

val_diurnal_pred = val_df.apply(lambda r: diurnal_lookup.get((r["dow"], r["hour"]), global_train_mean), axis=1).values
val_diurnal_mae = mean_absolute_error(val_df[TARGET], val_diurnal_pred)

print(f"Diurnal combinations learned: {len(diurnal_lookup)} (out of 168 max weekly slots)")
print(f"Diurnal Mean MAE (Validation): {val_diurnal_mae:.2f} kWh")

Step 7: Computing Diurnal Mean Baseline
Diurnal combinations learned: 168 (out of 168 max weekly slots)
Diurnal Mean MAE (Validation): 50.57 kWh


In [3]:
# Step 8: Run All Models on Test Set for Final Comparison
print("\n" + "="*60)
print("FINAL MODEL COMPARISON ON TEST SET")
print("="*60)

# 1. Persistence
test_persist_pred = test_df['usage_sum_last_1h'].values
test_persist_mae = mean_absolute_error(test_df[TARGET], test_persist_pred)
test_persist_rmse = mean_squared_error(test_df[TARGET], test_persist_pred, squared=False)

# 2. Diurnal Mean
test_diurnal_pred = test_df.apply(lambda r: diurnal_lookup.get((r["dow"], r["hour"]), global_train_mean), axis=1).values
test_diurnal_mae = mean_absolute_error(test_df[TARGET], test_diurnal_pred)
test_diurnal_rmse = mean_squared_error(test_df[TARGET], test_diurnal_pred, squared=False)

# 3. Ridge
X_test = scaler.transform(test_df[ALL_FEATURES].values)
test_ridge_pred = ridge.predict(X_test)
test_ridge_mae = mean_absolute_error(test_df[TARGET], test_ridge_pred)
test_ridge_rmse = mean_squared_error(test_df[TARGET], test_ridge_pred, squared=False)

# 4. LightGBM
test_lgb_mae = test_mae
test_lgb_rmse = test_rmse

models = ["Persistence", "Diurnal Mean", "Ridge", "LightGBM"]
maes = [test_persist_mae, test_diurnal_mae, test_ridge_mae, test_lgb_mae]
rmses = [test_persist_rmse, test_diurnal_rmse, test_ridge_rmse, test_lgb_rmse]

print(f"{'Model':<20} {'MAE (kWh)':>12} {'RMSE (kWh)':>12}")
print("-" * 46)
for m, a, r in zip(models, maes, rmses):
    print(f"{m:<20} {a:>12.2f} {r:>12.2f}")

best_idx = np.argmin(maes)
print(f"\nBest model on test MAE: {models[best_idx]} ({maes[best_idx]:.2f} kWh)")


FINAL MODEL COMPARISON ON TEST SET
Model                   MAE (kWh)   RMSE (kWh)
----------------------------------------------
Persistence                 38.76        77.76
Diurnal Mean                46.19        73.69
Ridge                       37.21        55.86
LightGBM                    15.92        32.58

Best model on test MAE: LightGBM (15.92 kWh)


In [4]:
# Step 9: Measure Single-Row Prediction Latency Percentiles
print("\n" + "="*60)
print("LEVEL 2: SINGLE-ROW INFERENCE LATENCY BENCHMARK")
print("="*60)

n_trials = 1000
single_row = test_df[ALL_FEATURES].iloc[0:1].values
latencies_ms = []

for _ in range(n_trials):
    t_start = time.perf_counter()
    _ = model.predict(single_row)
    latencies_ms.append((time.perf_counter() - t_start) * 1000.0)

mean_lat = np.mean(latencies_ms)
median_lat = np.median(latencies_ms)
p95_lat = np.percentile(latencies_ms, 95)
p99_lat = np.percentile(latencies_ms, 99)

print(f"Mean Latency:   {mean_lat:.3f} ms")
print(f"Median Latency: {median_lat:.3f} ms")
print(f"P95 Latency:    {p95_lat:.3f} ms")
print(f"P99 Latency:    {p99_lat:.3f} ms")
print(f"Requirement (< 100 ms): {'PASS' if p99_lat < 100 else 'FAIL'}")


LEVEL 2: SINGLE-ROW INFERENCE LATENCY BENCHMARK
Mean Latency:   0.261 ms
Median Latency: 0.119 ms
P95 Latency:    1.002 ms
P99 Latency:    2.058 ms
Requirement (< 100 ms): PASS


In [5]:
# Step 10: Group Alerts into Episodes & Evaluate Temporal Metrics
import os

print("\n" + "="*60)
print("ALERT EPISODE ANALYSIS & SYSTEM METRICS")
print("="*60)

def extract_episodes(binary_array):
    """Group consecutive positive flags into continuous index clusters."""
    episodes = []
    current_run = []
    for idx, is_pos in enumerate(binary_array):
        if is_pos:
            current_run.append(idx)
        elif current_run:
            episodes.append(set(current_run))
            current_run = []
    if current_run:
        episodes.append(set(current_run))
    return episodes

actual_episodes = extract_episodes(actual)
pred_episodes = extract_episodes(alerts)

# An actual surge event is caught if at least one predicted alert overlaps with it
detected_surges = sum(1 for true_ep in actual_episodes if any(len(true_ep & p_ep) > 0 for p_ep in pred_episodes))
ep_recall = detected_surges / max(len(actual_episodes), 1)

# A predicted alert event is valid if it hits an actual surge event
valid_alert_eps = sum(1 for p_ep in pred_episodes if any(len(p_ep & true_ep) > 0 for true_ep in actual_episodes))
ep_precision = valid_alert_eps / max(len(pred_episodes), 1)
ep_f1 = 2 * ep_precision * ep_recall / max(ep_precision + ep_recall, 1e-10)

print(f"Actual High-Load Episodes:    {len(actual_episodes)}")
print(f"Predicted Alert Episodes:     {len(pred_episodes)}")
print(f"Successfully Detected Events: {detected_surges}")
print(f"Episode-Level Precision:      {ep_precision:.4f}")
print(f"Episode-Level Recall:         {ep_recall:.4f}")
print(f"Episode-Level F1-Score:       {ep_f1:.4f}")

# Model Artifact Size on Disk (checked in artifacts folder)
model_path = ARTIFACTS_DIR / "lightgbm_model.txt"
model_size_kb = os.path.getsize(model_path) / 1024.0 if model_path.exists() else None
print(f"\nModel File Size: {model_size_kb:.2f} KB" if model_size_kb else "\nModel file not found!")

# Export consolidated test results to artifacts folder
results.update({
    "test_persist_mae": round(test_persist_mae, 4),
    "test_persist_rmse": round(test_persist_rmse, 4),
    "test_diurnal_mae": round(test_diurnal_mae, 4),
    "test_diurnal_rmse": round(test_diurnal_rmse, 4),
    "test_ridge_mae": round(test_ridge_mae, 4),
    "test_ridge_rmse": round(test_ridge_rmse, 4),
    "test_lgb_mae": round(test_lgb_mae, 4),
    "test_lgb_rmse": round(test_lgb_rmse, 4),
    "episode_precision": round(ep_precision, 4),
    "episode_recall": round(ep_recall, 4),
    "episode_f1": round(ep_f1, 4),
    "total_actual_episodes": len(actual_episodes),
    "total_pred_episodes": len(pred_episodes),
    "latency_median_ms": round(median_lat, 4),
    "latency_p95_ms": round(p95_lat, 4),
    "latency_p99_ms": round(p99_lat, 4),
    "model_size_kb": round(model_size_kb, 2) if model_size_kb else None
})

results_path = ARTIFACTS_DIR / "test_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved consolidated results to: {results_path.resolve()}")


ALERT EPISODE ANALYSIS & SYSTEM METRICS
Actual High-Load Episodes:    33
Predicted Alert Episodes:     49
Successfully Detected Events: 22
Episode-Level Precision:      0.4898
Episode-Level Recall:         0.6667
Episode-Level F1-Score:       0.5647

Model File Size: 1282.33 KB
Saved consolidated results to: C:\Users\LOQ\OneDrive\Desktop\IndustrialAI\steel-industry-energy-consumption\notebooks\artifacts\test_results.json
